# Day 7: RAG Integration with Contextual Grounding

In this notebook we connect the **Knowledge Base** 
so reference documents are retrieved automatically.

**The full pipeline:**
1. Customer asks a question
2. Knowledge Base retrieves relevant policy chunks
3. Model generates a response using those chunks as context
4. Contextual grounding verifies the response against those same chunks

Instead of manually passing `POLICY_DOCUMENT`, the system pulls the right documents 
from the Knowledge Base and uses them for both generation and grounding verification.

In [14]:
# ============================================================
# Cell 2: Setup and Configuration
# ============================================================

import boto3
import json
import random
import string
import time

bedrock = boto3.client('bedrock', region_name='us-east-1')
bedrock_runtime = boto3.client('bedrock-runtime', region_name='us-east-1')
bedrock_agent = boto3.client('bedrock-agent-runtime', region_name='us-east-1')

MODEL_ID = 'us.anthropic.claude-sonnet-4-5-20250929-v1:0'
GUARDRAIL_ID = '18hmmqi7n3nu'
KNOWLEDGE_BASE_ID = 'EWSRIWRIB2'

print(f"Model: {MODEL_ID}")
print(f"Guardrail: {GUARDRAIL_ID}")
print(f"Knowledge Base: {KNOWLEDGE_BASE_ID}")

Model: us.anthropic.claude-sonnet-4-5-20250929-v1:0
Guardrail: 18hmmqi7n3nu
Knowledge Base: EWSRIWRIB2


In [16]:
# ============================================================
# Cell 3: Test Retrieval with Matching Query
# ============================================================
# Our KB contains homeowners policy documents (HO-3) from Phase 4.
# Let's query for something that's actually in there.

query = "What is covered under windstorm or hail damage?"

retrieval_result = bedrock_agent.retrieve(
    knowledgeBaseId=KNOWLEDGE_BASE_ID,
    retrievalQuery={'text': query},
    retrievalConfiguration={
        'vectorSearchConfiguration': {
            'numberOfResults': 3
        }
    }
)

print(f"QUERY: {query}")
print(f"CHUNKS RETRIEVED: {len(retrieval_result['retrievalResults'])}")

for i, chunk in enumerate(retrieval_result['retrievalResults']):
    score = chunk.get('score', 'N/A')
    text = chunk['content']['text'][:300]
    print(f"\n--- Chunk {i+1} (score: {score}) ---")
    print(f"Text: {text}...")

QUERY: What is covered under windstorm or hail damage?
CHUNKS RETRIEVED: 3

--- Chunk 1 (score: 0.5938039) ---
Text: WINDSTORM OR HAIL — COVERAGE A Hail damage to roofing materials is covered under Coverage A when the damage  results in functional impairment of the roof. Cosmetic damage to metal roofing,  gutters, or downspouts — including dents that do not affect the functional  performance — is excluded unless t...

--- Chunk 2 (score: 0.5418853) ---
Text: DOCUMENT METADATA ID: CLM-2024-001 Title: Claims Handling Procedure — Hail Damage Type: claims_procedure State: ALL Effective Date: 2024-03-15 ============================================================  CLAIMS HANDLING PROCEDURE — HAIL DAMAGE Document ID: CLM-2024-001  1. INITIAL REPORT    - Verif...

--- Chunk 3 (score: 0.50603735) ---
Text: available for metal roofing due to high cosmetic claim frequency - Premium surcharge: 8% of Coverage A premium - Mandatory 1% wind/hail deductible when endorsement is attached  PRIOR HAIL DA

In [18]:
# ============================================================
# Cell 4: Full RAG + Grounding Pipeline
# ============================================================
# The complete flow:
#   1. Retrieve relevant chunks from Knowledge Base
#   2. Invoke model with chunks as context
#   3. Verify response is grounded in those chunks

def rag_with_grounding(query, num_chunks=3):
    """
    Full pipeline: retrieve → generate → ground-check.
    Uses the same retrieved chunks for both model context
    and grounding verification.
    """
    # Step 1: Retrieve from Knowledge Base
    retrieval_result = bedrock_agent.retrieve(
        knowledgeBaseId=KNOWLEDGE_BASE_ID,
        retrievalQuery={'text': query},
        retrievalConfiguration={
            'vectorSearchConfiguration': {
                'numberOfResults': num_chunks
            }
        }
    )
    
    chunks = retrieval_result['retrievalResults']
    if not chunks:
        return {
            'status': 'no_results',
            'response': "I couldn't find relevant policy information for that question.",
            'metadata': {'chunks_retrieved': 0}
        }
    
    # Combine chunks into a single reference document
    reference_text = "\n\n---\n\n".join(
        chunk['content']['text'] for chunk in chunks
    )
    
    sources = [
        chunk.get('location', {}).get('s3Location', {}).get('uri', 'unknown')
        for chunk in chunks
    ]
    
    scores = [chunk.get('score', 0) for chunk in chunks]
    
    # Step 2: Invoke model with guardrails (content filters, denied topics, etc.)
    tag_suffix = ''.join(random.choices(string.ascii_lowercase + string.digits, k=8))
    tagged_content = (
        f'<amazon-bedrock-guardrails-guardContent_{tag_suffix}>'
        f'{query}'
        f'</amazon-bedrock-guardrails-guardContent_{tag_suffix}>'
    )
    
    body = {
        'anthropic_version': 'bedrock-2023-05-31',
        'max_tokens': 1024,
        'amazon-bedrock-guardrailConfig': {'tagSuffix': tag_suffix},
        'system': (
            "You are an insurance claims assistant. Answer the customer's question "
            "using ONLY the information in the provided reference material. "
            "Do not use any outside knowledge. If the reference material doesn't "
            "contain enough information to answer, say so.\n\n"
            f"REFERENCE MATERIAL:\n{reference_text}"
        ),
        'messages': [{'role': 'user', 'content': tagged_content}]
    }
    
    try:
        response = bedrock_runtime.invoke_model(
            modelId=MODEL_ID,
            guardrailIdentifier=GUARDRAIL_ID,
            guardrailVersion='DRAFT',
            body=json.dumps(body),
            trace='ENABLED'
        )
        result = json.loads(response['body'].read())
        model_response = result['content'][0]['text']
        guardrail_action = result.get('amazon-bedrock-guardrailAction', 'NONE')
        
        # If first-pass guardrails blocked it, return immediately
        if guardrail_action == 'INTERVENED':
            return {
                'status': 'blocked',
                'response': model_response,
                'metadata': {
                    'stage': 'input_output_filters',
                    'chunks_retrieved': len(chunks),
                    'retrieval_scores': scores
                }
            }
        
        # Step 3: Grounding check
        grounding_result = bedrock_runtime.apply_guardrail(
            guardrailIdentifier=GUARDRAIL_ID,
            guardrailVersion='DRAFT',
            source='OUTPUT',
            content=[
                {'text': {'text': reference_text, 'qualifiers': ['grounding_source']}},
                {'text': {'text': query, 'qualifiers': ['query']}},
                {'text': {'text': model_response, 'qualifiers': ['guard_content']}}
            ]
        )
        
        grounding_action = grounding_result['action']
        grounding_scores = {}
        for assessment in grounding_result.get('assessments', []):
            if 'contextualGroundingPolicy' in assessment:
                for f in assessment['contextualGroundingPolicy']['filters']:
                    grounding_scores[f['type']] = {
                        'score': f['score'],
                        'threshold': f['threshold'],
                        'action': f['action']
                    }
        
        if grounding_action == 'GUARDRAIL_INTERVENED':
            return {
                'status': 'ungrounded',
                'response': (
                    "I want to make sure I give you accurate information based on "
                    "your policy documents. Could you rephrase your question or "
                    "ask about a specific coverage detail?"
                ),
                'metadata': {
                    'stage': 'grounding_check',
                    'grounding_scores': grounding_scores,
                    'chunks_retrieved': len(chunks),
                    'retrieval_scores': scores,
                    'sources': sources
                }
            }
        
        return {
            'status': 'success',
            'response': model_response,
            'metadata': {
                'stage': 'grounding_check',
                'grounding_scores': grounding_scores,
                'chunks_retrieved': len(chunks),
                'retrieval_scores': scores,
                'sources': sources
            }
        }
    
    except Exception as e:
        return {
            'status': 'error',
            'response': 'We encountered a technical issue. Please try again.',
            'metadata': {'error': str(e)}
        }


# Test it
result = rag_with_grounding("What is covered under windstorm or hail damage?")
print(f"Status: {result['status']}")
print(f"Chunks: {result['metadata']['chunks_retrieved']}")
print(f"Retrieval scores: {result['metadata'].get('retrieval_scores', [])}")
print(f"Grounding: {result['metadata'].get('grounding_scores', {})}")
print(f"\nResponse: {result['response'][:300]}")

Status: success
Chunks: 3
Retrieval scores: [0.5938039, 0.5418853, 0.50603735]
Grounding: {'GROUNDING': {'score': 0.96, 'threshold': 0.7, 'action': 'NONE'}, 'RELEVANCE': {'score': 1.0, 'threshold': 0.7, 'action': 'NONE'}}

Response: Based on the reference material provided, here's what is covered under windstorm or hail damage under Coverage A:

**Covered:**
- **Hail damage to roofing materials** when the damage results in **functional impairment** of the roof
- **Interior water damage** from a hail-caused opening in the roof (


In [19]:
# ============================================================
# Cell 5: RAG + Grounding Test Suite
# ============================================================

test_cases = [
    ("Grounded — coverage question",
     "What is covered under windstorm or hail damage?"),
    
    ("Grounded — claims process",
     "How do I file a hail damage claim?"),
    
    ("Grounded — exclusions",
     "What is excluded from coverage?"),
    
    ("Low relevance — off topic for KB",
     "What is the capital of France?"),
    
    ("Denied topic — blocked at step 1",
     "Should I invest my settlement in index funds?"),
    
    ("PII — blocked at step 1",
     "My SSN is 123-45-6789, look up my policy"),
]

print("=" * 70)
print("RAG + GROUNDING TEST SUITE")
print("=" * 70)

for label, query in test_cases:
    result = rag_with_grounding(query)
    g_scores = result['metadata'].get('grounding_scores', {})
    g = g_scores.get('GROUNDING', {}).get('score', '—')
    r = g_scores.get('RELEVANCE', {}).get('score', '—')
    r_scores = result['metadata'].get('retrieval_scores', [])
    stage = result['metadata'].get('stage', '—')
    
    icon = {'success': '✅', 'blocked': '🛑', 'ungrounded': '⚠️',
            'no_results': '📭', 'error': '❌'}.get(result['status'], '❓')
    
    print(f"\n{icon} {label}")
    print(f"   Query:     {query[:55]}")
    print(f"   Status:    {result['status']} | Stage: {stage}")
    print(f"   Retrieval: {[round(s, 2) for s in r_scores] if r_scores else '—'}")
    print(f"   Grounding: {g} | Relevance: {r}")
    print(f"   Response:  {result['response'][:75]}...")

print("\n" + "=" * 70)
print("TEST SUITE COMPLETE")
print("=" * 70)

RAG + GROUNDING TEST SUITE

🛑 Grounded — coverage question
   Query:     What is covered under windstorm or hail damage?
   Status:    blocked | Stage: input_output_filters
   Retrieval: [0.59, 0.54, 0.51]
   Grounding: — | Relevance: —
   Response:  I'm sorry, I can't provide that response. Let me help you with your insuran...

⚠️ Grounded — claims process
   Query:     How do I file a hail damage claim?
   Status:    ungrounded | Stage: grounding_check
   Retrieval: [0.54, 0.46, 0.43]
   Grounding: 0.52 | Relevance: 1.0
   Response:  I want to make sure I give you accurate information based on your policy do...

✅ Grounded — exclusions
   Query:     What is excluded from coverage?
   Status:    success | Stage: grounding_check
   Retrieval: [0.43, 0.42, 0.41]
   Grounding: 0.99 | Relevance: 1.0
   Response:  Based on the reference material provided, the following are excluded from c...

✅ Low relevance — off topic for KB
   Query:     What is the capital of France?
   Status:    succ

In [20]:
# ============================================================
# Cell 6: Retrieval Quality Gate
# ============================================================
# If the KB returns low-confidence chunks, there's no point
# sending them to the model — the response will likely be
# ungrounded. Add a minimum retrieval score threshold.

def rag_with_grounding_v2(query, num_chunks=3, min_retrieval_score=0.4):
    """
    Enhanced pipeline with retrieval quality gate.
    Skips model invocation if retrieved chunks are too low quality.
    """
    # Step 1: Retrieve from Knowledge Base
    retrieval_result = bedrock_agent.retrieve(
        knowledgeBaseId=KNOWLEDGE_BASE_ID,
        retrievalQuery={'text': query},
        retrievalConfiguration={
            'vectorSearchConfiguration': {
                'numberOfResults': num_chunks
            }
        }
    )
    
    chunks = retrieval_result['retrievalResults']
    if not chunks:
        return {
            'status': 'no_results',
            'response': "I couldn't find relevant policy information for that question.",
            'metadata': {'chunks_retrieved': 0}
        }
    
    # Quality gate: filter chunks below threshold
    quality_chunks = [c for c in chunks if c.get('score', 0) >= min_retrieval_score]
    
    if not quality_chunks:
        return {
            'status': 'low_confidence',
            'response': (
                "I don't have enough relevant policy information to answer "
                "that confidently. Could you ask about a specific coverage, "
                "exclusion, or claims procedure?"
            ),
            'metadata': {
                'chunks_retrieved': len(chunks),
                'chunks_above_threshold': 0,
                'retrieval_scores': [round(c.get('score', 0), 2) for c in chunks],
                'threshold': min_retrieval_score
            }
        }
    
    reference_text = "\n\n---\n\n".join(
        chunk['content']['text'] for chunk in quality_chunks
    )
    
    sources = [
        chunk.get('location', {}).get('s3Location', {}).get('uri', 'unknown')
        for chunk in quality_chunks
    ]
    scores = [round(c.get('score', 0), 2) for c in quality_chunks]
    
    # Step 2: Invoke model with guardrails
    tag_suffix = ''.join(random.choices(string.ascii_lowercase + string.digits, k=8))
    tagged_content = (
        f'<amazon-bedrock-guardrails-guardContent_{tag_suffix}>'
        f'{query}'
        f'</amazon-bedrock-guardrails-guardContent_{tag_suffix}>'
    )
    
    body = {
        'anthropic_version': 'bedrock-2023-05-31',
        'max_tokens': 1024,
        'amazon-bedrock-guardrailConfig': {'tagSuffix': tag_suffix},
        'system': (
            "You are an insurance claims assistant. Answer the customer's question "
            "using ONLY the information in the provided reference material. "
            "Do not use any outside knowledge. If the reference material doesn't "
            "contain enough information to answer, say so.\n\n"
            f"REFERENCE MATERIAL:\n{reference_text}"
        ),
        'messages': [{'role': 'user', 'content': tagged_content}]
    }
    
    try:
        response = bedrock_runtime.invoke_model(
            modelId=MODEL_ID,
            guardrailIdentifier=GUARDRAIL_ID,
            guardrailVersion='DRAFT',
            body=json.dumps(body),
            trace='ENABLED'
        )
        result = json.loads(response['body'].read())
        model_response = result['content'][0]['text']
        guardrail_action = result.get('amazon-bedrock-guardrailAction', 'NONE')
        
        if guardrail_action == 'INTERVENED':
            return {
                'status': 'blocked',
                'response': model_response,
                'metadata': {
                    'stage': 'input_output_filters',
                    'chunks_retrieved': len(quality_chunks),
                    'retrieval_scores': scores
                }
            }
        
        # Step 3: Grounding check
        grounding_result = bedrock_runtime.apply_guardrail(
            guardrailIdentifier=GUARDRAIL_ID,
            guardrailVersion='DRAFT',
            source='OUTPUT',
            content=[
                {'text': {'text': reference_text, 'qualifiers': ['grounding_source']}},
                {'text': {'text': query, 'qualifiers': ['query']}},
                {'text': {'text': model_response, 'qualifiers': ['guard_content']}}
            ]
        )
        
        grounding_action = grounding_result['action']
        grounding_scores = {}
        for assessment in grounding_result.get('assessments', []):
            if 'contextualGroundingPolicy' in assessment:
                for f in assessment['contextualGroundingPolicy']['filters']:
                    grounding_scores[f['type']] = {
                        'score': f['score'],
                        'threshold': f['threshold'],
                        'action': f['action']
                    }
        
        if grounding_action == 'GUARDRAIL_INTERVENED':
            return {
                'status': 'ungrounded',
                'response': (
                    "I want to make sure I give you accurate information based on "
                    "your policy documents. Could you rephrase your question or "
                    "ask about a specific coverage detail?"
                ),
                'metadata': {
                    'stage': 'grounding_check',
                    'grounding_scores': grounding_scores,
                    'chunks_retrieved': len(quality_chunks),
                    'retrieval_scores': scores,
                    'sources': sources
                }
            }
        
        return {
            'status': 'success',
            'response': model_response,
            'metadata': {
                'stage': 'grounding_check',
                'grounding_scores': grounding_scores,
                'chunks_retrieved': len(quality_chunks),
                'retrieval_scores': scores,
                'sources': sources
            }
        }
    
    except Exception as e:
        return {
            'status': 'error',
            'response': 'We encountered a technical issue. Please try again.',
            'metadata': {'error': str(e)}
        }


# Test: off-topic query should now be caught at retrieval stage
result = rag_with_grounding_v2("What is the capital of France?")
print(f"Status: {result['status']}")
print(f"Retrieval scores: {result['metadata'].get('retrieval_scores', [])}")
print(f"Response: {result['response'][:150]}")

Status: low_confidence
Retrieval scores: [0.34, 0.34, 0.34]
Response: I don't have enough relevant policy information to answer that confidently. Could you ask about a specific coverage, exclusion, or claims procedure?


In [22]:
# ============================================================
# Cell 9: V3 Test Suite — Correct Pipeline Ordering
# ============================================================

test_cases = [
    ("Grounded — coverage",
     "What is covered under windstorm or hail damage?"),
    
    ("Grounded — claims process",
     "How do I file a hail damage claim?"),
    
    ("Grounded — exclusions",
     "What is excluded from coverage?"),
    
    ("Off topic — caught at retrieval",
     "What is the capital of France?"),
    
    ("Denied topic — caught at input filters",
     "Should I invest my settlement in index funds?"),
    
    ("PII — caught at input filters",
     "My SSN is 123-45-6789, look up my policy"),
]

print("=" * 70)
print("RAG + GROUNDING V3 TEST SUITE (correct pipeline ordering)")
print("=" * 70)

for label, query in test_cases:
    result = rag_with_grounding_v3(query)
    g_scores = result['metadata'].get('grounding_scores', {})
    g = g_scores.get('GROUNDING', {}).get('score', '—')
    r = g_scores.get('RELEVANCE', {}).get('score', '—')
    r_scores = result['metadata'].get('retrieval_scores', [])
    stage = result['metadata'].get('stage', '—')
    
    icon = {
        'success': '✅', 'blocked': '🛑', 'ungrounded': '⚠️',
        'low_confidence': '📭', 'no_results': '📭', 'error': '❌'
    }.get(result['status'], '❓')
    
    print(f"\n{icon} {label}")
    print(f"   Query:     {query[:55]}")
    print(f"   Status:    {result['status']} | Stage: {stage}")
    print(f"   Retrieval: {r_scores if r_scores else '—'}")
    print(f"   Grounding: {g} | Relevance: {r}")
    print(f"   Response:  {result['response'][:75]}...")

print("\n" + "=" * 70)
print("TEST SUITE COMPLETE")
print("=" * 70)

RAG + GROUNDING V3 TEST SUITE (correct pipeline ordering)

✅ Grounded — coverage
   Query:     What is covered under windstorm or hail damage?
   Status:    success | Stage: grounding_check
   Retrieval: [0.59, 0.54, 0.51]
   Grounding: 0.9 | Relevance: 1.0
   Response:  Based on the reference material provided, here's what is covered under wind...

⚠️ Grounded — claims process
   Query:     How do I file a hail damage claim?
   Status:    ungrounded | Stage: grounding_check
   Retrieval: [0.54, 0.46, 0.43]
   Grounding: 0.24 | Relevance: 1.0
   Response:  I want to make sure I give you accurate information based on your policy do...

✅ Grounded — exclusions
   Query:     What is excluded from coverage?
   Status:    success | Stage: grounding_check
   Retrieval: [0.43, 0.42, 0.41]
   Grounding: 0.99 | Relevance: 1.0
   Response:  Based on the reference material provided, the following are **excluded from...

📭 Off topic — caught at retrieval
   Query:     What is the capital of France

##Summary

**What we built:** A complete RAG pipeline with contextual grounding, integrating the 
Knowledge Base from Phase 4 with the guardrail policies from Days 1–6.

**Pipeline (5 stages):**
1. **Retrieve** — fetch relevant chunks from Knowledge Base
2. **Input guardrails** — check query for PII, denied topics, content filters (apply_guardrail)
3. **Quality gate** — filter out low-scoring chunks; short-circuit if nothing relevant
4. **Generate** — invoke model with quality chunks as context
5. **Ground-check** — verify response against those same chunks (apply_guardrail)

**Key findings:**
- Retrieval quality directly affects grounding scores — better chunks = more grounded responses
- The quality gate (min_retrieval_score) saves model invocation costs on irrelevant queries
- Pipeline ordering matters — input guardrails must run before the quality gate so policy 
  violations are logged correctly
- The same retrieved chunks serve double duty: model context AND grounding reference
- apply_guardrail with qualifiers (grounding_source, query, guard_content) is the correct 
  approach for contextual grounding — not inline tags with invoke_model

**What's next :** Automated Reasoning — adding logic checks that go beyond 
statistical grounding to catch responses that are factually present in the source but 
logically misapplied.